# Patrones de desarrollo mundial — PCA y Clustering sobre datos WDI

**Trabajo Práctico Final · Análisis Multivariado y Descubrimiento de Patrones**  
Universidad Austral · 2024

---

### Resumen ejecutivo

Este trabajo analiza **191 países** usando 14 indicadores del Banco Mundial (WDI) para responder tres preguntas:

1. ¿Qué dimensiones subyacentes organizan el desarrollo global?
2. ¿Existen grupos naturales de países con perfiles similares?
3. ¿Cómo evolucionaron esos grupos entre 2005 y 2023?

**Hallazgos principales:**
- Una sola dimensión (PC1, *gradiente de desarrollo*) explica el **41 %** de la varianza total.
- Dos grupos robustos emergen del clustering: *Desarrollados* (92 países) y *En desarrollo* (99 países).
- Entre 2005 y 2023, **40 países graduaron** al grupo más desarrollado; ninguno retrocedió. El avance fue impulsado principalmente por **acceso a Internet** (47 %) y **esperanza de vida** (22 %), no por el PBI.

In [ ]:
import subprocess, sys, json
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

HERE = Path().resolve()
if HERE.name != 'Benja':
    HERE = HERE / 'Benja'
ROOT = HERE.parent
PROC = ROOT / 'data' / 'processed'
FIG  = HERE / 'figures'

result = subprocess.run(
    [sys.executable, str(HERE / 'figures_mejoradas.py')],
    capture_output=True, text=True, cwd=str(ROOT)
)
print(result.stdout if result.returncode == 0 else result.stderr)

---
## 1 · Datos y variables

Usamos el **World Development Indicators (WDI)** del Banco Mundial. Se seleccionaron dos cortes temporales:

| Corte | Año | Motivo |
|---|---|---|
| Moderno | **2023** | Año estable post-COVID con cobertura ≥ 78 % en todas las variables |
| Temprano | **2005** | Año más antiguo donde todas las variables superan el umbral de cobertura del 75 % |

Se descartaron tres variables por cobertura insuficiente o redundancia: Índice de Gini, Gasto en educación e INB PPA (*r* ≈ 0,98 con PBI per cápita).

| Dimensión | Variables |
|---|---|
| Ingreso y crecimiento | PBI per cápita, Crecimiento del PBI |
| Precios y empleo | Inflación, Desempleo |
| Estructura económica | Cap. fijo, Exportaciones, Importaciones, Agricultura, Industria, Servicios |
| Desarrollo humano | Urbanización, Esperanza de vida, Internet |
| Medioambiente | Emisiones de CO₂ per cápita |

---
## 2 · Análisis exploratorio

### Transformaciones aplicadas

Antes de modelar se corrigió la asimetría de cada variable:

| Transformación | Variables |
|---|---|
| Logaritmo natural | PBI per cápita (asimetría 2,2 → ~0) |
| log(1 + x) | Emisiones CO₂ (asimetría 5,4 → 0,5) |
| Yeo-Johnson | Inflación, Desempleo, Exportaciones, Importaciones, Agricultura, Industria |
| Sin transformar | Crecimiento PBI, Cap. Fijo, Servicios, Urbanización, Esp. vida, Internet |

Luego se estandarizó todo con `StandardScaler` (media = 0, desviación = 1).

### Correlaciones entre variables

In [ ]:
display(Image(filename=str(FIG / 'fig_correlacion.png'), width=900))

> **Cómo leer este gráfico:** triángulo superior = coeficiente de Spearman (ρ); azul = positivo, rojo = negativo; más intenso = más fuerte. Diagonal = distribución de cada variable.
>
> **Qué observar:** bloque azul entre PBI, Internet, Esperanza de vida, Urbanización y CO₂ — todas miden el mismo fenómeno subyacente. Agricultura correlaciona negativamente con todas. Esta estructura es exactamente lo que capturará PC1.

---
## 3 · Análisis de Componentes Principales (PCA)

### ¿Cuántos componentes retener?

Se usó el **Análisis Paralelo de Horn**: se comparan los autovalores observados contra los de 1.000 matrices aleatorias del mismo tamaño. Solo se retienen los componentes que superan el percentil 95 del ruido puro.

In [ ]:
info = json.loads((PROC / 'pca_info_2023.json').read_text(encoding='utf-8'))
df_ev = pd.read_csv(PROC / 'pca_autovalores_2023.csv')
print(f"Kaiser (autovalor > 1):          {info['kaiser']} componentes")
print(f"80 % varianza acumulada:         {info['n_80pct']} componentes")
print(f"Análisis paralelo de Horn (p95): {info['parallel_analysis']} componentes  <- ADOPTADO")
print(f"Varianza acumulada con 3 PCs:    {df_ev['var_acumulada'].iloc[2]*100:.1f} %")

In [ ]:
display(Image(filename=str(FIG / 'fig_scree.png'), width=820))

> Barras azul oscuro = componentes retenidos. Los triángulos rojos marcan el umbral de Horn: por encima hay señal real, por debajo hay ruido. Tres componentes explican el 64,9 % de la varianza total.

### Cargas por componente

In [ ]:
display(Image(filename=str(FIG / 'fig_loadings.png'), width=920))

> **PC1 (41 %):** Internet, Esperanza de vida, PBI y Urbanización (positivos) contra Agricultura (negativo). Es el eje del desarrollo integral.  
> **PC2 (13 %):** opone países industrializados a los orientados a servicios.  
> **PC3 (10 %):** diferencia economías abiertas al comercio internacional.

### Círculo de correlaciones

In [ ]:
display(Image(filename=str(FIG / 'fig_circulo_cargas.png'), width=750))

> **Cómo leer este gráfico:** cada flecha es una variable. La longitud indica qué tan bien está representada por los dos primeros componentes (más cerca del círculo exterior = mejor). La dirección indica su relación con PC1 (horizontal) y PC2 (vertical). Azul = correlación positiva con PC1; rojo = negativa.
>
> Variables como Internet, Esperanza de vida y PBI apuntan a la derecha (PC1 alto = desarrollo) mientras Agricultura apunta a la izquierda (países menos industrializados). Industria apunta hacia arriba (PC2 alto = estructura industrial).

---
## 4 · Clustering

El clustering se realizó sobre las **14 variables originales** (no sobre el espacio PCA) para evitar el sesgo del *tandem analysis*. Como verificación: las particiones sobre variables completas y sobre 3 PCs son **idénticas** (ARI = 1,00).

### Selección del número de clusters

In [ ]:
display(Image(filename=str(FIG / 'fig_metricas_k.png'), width=900))

> Los cuatro criterios señalan k = 2. La estabilidad bootstrap (criterio de Hennig) confirma: Jaccard 0,96 y 0,94 por cluster — muy por encima del umbral de 0,75.

### Resultado: dos grupos

In [ ]:
display(Image(filename=str(FIG / 'fig_clusters_k2.png'), width=900))

> **Rojo — En desarrollo (n ≈ 99):** bajo PBI, menor esperanza de vida, baja cobertura de Internet, alta participación agrícola.  
> **Azul — Desarrollado (n ≈ 92):** economías de alta productividad, economías orientadas a servicios, acceso masivo a Internet.
>
> ⚠️ El desarrollo es un continuo: el silhouette moderado (0,28) confirma que no hay saltos nítidos en los datos. Estos grupos son la partición más informativa del gradiente, no categorías rígidas.

In [ ]:
display(Image(filename=str(FIG / 'fig_clusters_k3.png'), width=900))

> Con k = 3 emerge un **grupo intermedio** (naranja) que captura países en transición: ingreso medio con industrialización creciente.

In [ ]:
cruce = pd.read_csv(PROC / 'cruce_ingreso_k2_2023.csv', index_col=0)
cruce.index = ['En desarrollo', 'Desarrollado']
print('Cruce cluster × nivel de ingreso:')
display(cruce)
res  = json.loads((PROC / 'clustering_resultados_2023.json').read_text(encoding='utf-8'))
chi2 = res['perfilado'][2]['chi2_ingreso']['chi2']
p    = res['perfilado'][2]['chi2_ingreso']['p']
print(f'chi2 = {chi2:.1f}  (p = {p:.2e}) — asociacion altamente significativa')

In [ ]:
display(Image(filename=str(FIG / 'fig_silhouette_k2.png'), width=740))

---
## 5 · Evolución temporal 2005 → 2023

Para comparar países en el tiempo se construye un **espacio PCA común**: ambos años apilados, un solo escalado y un solo PCA ajustados sobre los datos combinados.

### Mayores avances y retrocesos

In [ ]:
display(Image(filename=str(FIG / 'fig_trayectorias.png'), width=900))

> Cada flecha va del punto gris (posición 2005) al punto coloreado (posición 2023). Verde = avanzó en el gradiente de desarrollo. Rojo = retrocedió. Los retrocesos más marcados coinciden con crisis documentadas: colapso económico (Venezuela), conflicto armado (Sudán) y crisis financiera (Líbano).

### Flujo entre grupos — diagrama de Sankey

In [ ]:
display(Image(filename=str(FIG / 'fig_transicion_sankey.png'), width=820))

> Cada banda muestra el flujo de países entre 2005 (izquierda) y 2023 (derecha). El ancho es proporcional al número de países. **40 países** (banda verde) pasaron del grupo en desarrollo al desarrollado. **Ningún país** hizo el camino inverso.

### Distribución por grupo — gráfico de waffle

In [ ]:
display(Image(filename=str(FIG / 'fig_transicion_waffle.png'), width=820))

> Cada cuadrado = 1 país. La expansión del área azul de 2005 a 2023 representa los 40 países que graduaron al grupo más desarrollado.

### ¿Qué impulsó el avance?

In [ ]:
display(Image(filename=str(FIG / 'fig_decomposicion.png'), width=800))

> **El avance fue liderado por conectividad (47 %) y salud (22 %), no por el PBI (9 %).** El PBI usado es en dólares constantes 2015 (real), por lo que el resultado no es un artefacto inflacionario.

---
## 6 · Conclusiones

1. **El desarrollo se organiza como un gradiente unidimensional.** PC1 (41 % de la varianza) integra ingreso, salud, conectividad y estructura económica en un único eje. El análisis paralelo de Horn confirma que la señal es real.

2. **Dos grupos robustos y replicables.** K-means sobre 14 variables identifica clusters con Jaccard > 0,94. Los grupos coinciden ampliamente con la clasificación del Banco Mundial, con excepciones informativas.

3. **Progreso generalizado impulsado por salud y tecnología.** Internet (47 %) y esperanza de vida (22 %) lideraron el avance entre 2005 y 2023. El crecimiento del PBI explicó solo el 9 %.

4. **El avance no es universal ni irreversible.** 40 países graduaron de grupo; ninguno retrocedió. Pero Líbano, Venezuela y Sudán retrocedieron sustancialmente, evidenciando que el progreso puede revertirse ante crisis severas.

---
*Figuras generadas con `python Benja/figures_mejoradas.py` · Pipeline completo en `python src/run_all.py`*